# Synthetic Probe Analysis — Training

Train logistic-regression probes on JudgementLM internal representations collected
from the **synthetic probe dataset** (`data/{dataset}/probe_dataset.json`).

**Labels**: `label` field in the synthetic dataset (`"valid"` / `"invalid"`)

---

**Part I — Attention Head Probe**  
Score every (layer, head) pair on synthetic 5-fold CV, select the best `TOP_K`
heads by F1, then train a final logistic-regression probe on the full synthetic
training set.  The fitted probe is saved to disk for evaluation in
`synthetic_probe_test.ipynb`.

**Part II — Layer Output Probe (legacy)**  
Score every layer's full residual-stream vector on synthetic 5-fold CV, select
the best layer, and train a final probe.

---

**Metrics reported during CV:** AUROC · F1 · Accuracy

In [1]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from analysis.loaders import (
    load_synthetic_activations, load_synthetic_layer_outputs, load_synthetic_responses,
)
from scholarlm.utils.probe import grouped_kfold_split, grouped_holdout_split
import paths

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "legend.title_fontsize": 9,
    "axes.linewidth": 0.6,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "lines.linewidth": 1.2, "lines.markersize": 4,
    "legend.frameon": False,
    "figure.dpi": 150, "savefig.dpi": 300,
    "savefig.format": "pdf", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

FIGURES_DIR = "../figures/synthetic_probe/"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
# ── Parameters ───────────────────────────────────────────────────────────────
DATASET        = 'pond'
JUDGE_MODEL    = 'llama-3.1-8b'   # must match the judge used for the synthetic run
JUDGE_DATE_SYN = '2026_04_30'           # auto-detect latest synthetic probe run

TOP_K   = 1    # number of attention heads for the final probe
N_FOLDS = 5

## Load Synthetic Data

In [4]:
syn_activations = load_synthetic_activations(DATASET, JUDGE_MODEL, JUDGE_DATE_SYN)
syn_responses   = load_synthetic_responses(DATASET, JUDGE_MODEL, JUDGE_DATE_SYN)
syn_df          = pd.DataFrame(syn_responses)

syn_measurement_ids = syn_df['measurement_id'].tolist()
syn_labels          = (syn_df['label'] == 'valid').to_numpy(dtype=bool)
syn_groups          = syn_df['document_id'].to_numpy()

print(f'Synthetic activations : {len(syn_activations.files)} entries')
print(f'Synthetic records     : {len(syn_df)}')
print(f'  Valid   : {syn_labels.sum()} ({syn_labels.mean():.1%})')
print(f'  Invalid : {(~syn_labels).sum()} ({(~syn_labels).mean():.1%})')
print(f'Unique papers in synthetic set : {syn_df["document_id"].nunique()}')
print()
print('Modification types:')
print(syn_df['modification_type'].value_counts(dropna=False))

Synthetic activations : 5274 entries
Synthetic records     : 5274
  Valid   : 1758 (33.3%)
  Invalid : 3516 (66.7%)
Unique papers in synthetic set : 49

Modification types:
modification_type
NaN                 1758
table_value          879
change_attribute     472
change_value         460
change_entity        455
noise_value          440
noise_entity         439
change_units         371
Name: count, dtype: int64


In [5]:
# Use all synthetic data for training (no calibration holdout).
# Group-aware split ensures no paper appears in both train and CV test folds.
syn_train_idx, _, _ = grouped_holdout_split(
    syn_groups, train_frac=1.0, cal_frac=0.0, random_state=42
)
syn_cv_idx    = syn_train_idx
syn_labels_cv = syn_labels[syn_cv_idx]
syn_groups_cv = syn_groups[syn_cv_idx]
kfold_cv = list(grouped_kfold_split(syn_groups_cv, n_splits=N_FOLDS, random_state=42))

print(f'Training set : n={len(syn_train_idx)} ({len(syn_train_idx)/len(syn_labels):.1%})  pos={syn_labels[syn_train_idx].mean():.2%}')

Training set : n=5274 (100.0%)  pos=33.33%


---
## Part I — Attention Head Probe

Score every (layer, head) pair by F1 on synthetic 5-fold CV, select the best
`TOP_K` heads, then train a final probe on the full synthetic training set and
save it to disk.

## Build Head Feature Matrices

In [6]:
_arr0 = np.array(syn_activations[str(syn_measurement_ids[0])], dtype=np.float32)
n_layers, n_heads, head_dim = _arr0.shape
print(f'Activation shape : n_layers={n_layers}, n_heads={n_heads}, head_dim={head_dim}')
print(f'Total heads      : {n_layers * n_heads}')

print('\nLoading synthetic activations into memory...')
_all_syn = {
    str(mid): np.array(syn_activations[str(mid)], dtype=np.float32)
    for mid in syn_measurement_ids
}
print('Building per-head feature matrices...')
head_datasets_syn: dict[tuple[int, int], np.ndarray] = {}
for l in range(n_layers):
    for h in range(n_heads):
        head_datasets_syn[(l, h)] = np.stack(
            [_all_syn[str(mid)][l, h, :] for mid in syn_measurement_ids], axis=0
        )
del _all_syn
print(f'  Done. Shape: {head_datasets_syn[(0, 0)].shape}')

Activation shape : n_layers=32, n_heads=32, head_dim=128
Total heads      : 1024

Loading synthetic activations into memory...
Building per-head feature matrices...
  Done. Shape: (5274, 128)


In [7]:
def cv_score(probe, X, y, kfold_cv):
    fold_accs, fold_aurocs, fold_f1s, fold_bs = [], [], [], []
    fold_test_idx, fold_probs_list = [], []
    for train_idx, test_idx in kfold_cv:
        probe.fit(X[train_idx], y[train_idx])
        y_pred = probe.predict(X[test_idx])
        y_true = y[test_idx]
        probs  = probe.predict_proba(X[test_idx])[:, 1]
        fold_accs.append(float((y_pred == y_true).mean()))
        auroc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (~y_true).sum() > 0 else float('nan')
        fold_aurocs.append(auroc)
        tp   = int(( y_pred &  y_true).sum())
        fp   = int(( y_pred & ~y_true).sum())
        fn   = int((~y_pred &  y_true).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fold_f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0)
        fold_bs.append(float(brier_score_loss(y_true, probs)))
        fold_test_idx.extend(test_idx)
        fold_probs_list.append(probs)
    oof_probs  = np.empty(len(y))
    sort_order = np.argsort(fold_test_idx)
    oof_probs[np.array(fold_test_idx)[sort_order]] = np.concatenate(fold_probs_list)[sort_order]
    return (
        float(np.mean(fold_accs)),      list(fold_accs),
        float(np.nanmean(fold_aurocs)), list(fold_aurocs),
        float(np.mean(fold_f1s)),       list(fold_f1s),
        float(np.mean(fold_bs)),        list(fold_bs),
        oof_probs,
    )

## Head Selection (CV on Synthetic Data)

In [8]:
probe_template = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0, class_weight='balanced', solver='lbfgs',
        max_iter=1000, random_state=42,
    ))
])

head_scores_acc   = np.zeros((n_layers, n_heads))
head_scores_auroc = np.zeros((n_layers, n_heads))
head_scores_f1    = np.zeros((n_layers, n_heads))

print(f'Scoring {n_layers * n_heads} heads on synthetic CV pool (n={len(syn_cv_idx)})...')
for l in range(n_layers):
    for h in range(n_heads):
        X_cv = head_datasets_syn[(l, h)][syn_cv_idx]
        (mean_acc, _,
         mean_auroc, _,
         mean_f1, _,
         mean_bs, _,
         _) = cv_score(probe_template, X_cv, syn_labels_cv, kfold_cv)
        head_scores_acc[l, h]   = mean_acc
        head_scores_auroc[l, h] = mean_auroc
        head_scores_f1[l, h]    = mean_f1
    if (l + 1) % 8 == 0:
        print(f'  Layer {l + 1}/{n_layers} done')
print('Done.')

Scoring 1024 heads on synthetic CV pool (n=5274)...
  Layer 8/32 done
  Layer 16/32 done
  Layer 24/32 done
  Layer 32/32 done
Done.


In [ ]:
fname = f'head_scores_{DATASET}_{JUDGE_MODEL}.npy'
score_mat = head_scores_f1
np.save(fname, score_mat)
score_mat = np.load(fname)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3))
im = ax.imshow(np.sort(score_mat, axis=1), cmap='magma', aspect='auto')
ax.set_xlabel('Head (sorted by F1)')
ax.set_ylabel('Layer')
ax.set_title(f'{DATASET} / {JUDGE_MODEL}')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
#fig.savefig(FIGURES_DIR + f'synprobe_heatmap_{DATASET}_{JUDGE_MODEL}.pdf', bbox_inches='tight', dpi=100)

In [51]:
flat_indices = np.argsort(score_mat.flatten())[-TOP_K:][::-1]
top_k_heads  = [np.unravel_index(idx, score_mat.shape) for idx in flat_indices]

print(f'Top {TOP_K} head(s) by CV F1:')
for i, (bl, bh) in enumerate(top_k_heads):
    print(f'  {i+1}. layer={bl}, head={bh}  '
          f'AUROC={head_scores_auroc[bl, bh]:.4f}  '
          f'F1={head_scores_f1[bl, bh]:.4f}  '
          f'Acc={head_scores_acc[bl, bh]:.4f}')

Top 1 head(s) by CV F1:
  1. layer=14, head=29  AUROC=0.6393  F1=0.5373  Acc=0.5710


### Train Final Probe and Save

Train the final probe on the full synthetic training set using the selected
`TOP_K` heads, then save it to disk.  The saved file includes the fitted probe,
the selected head indices, the training prevalence (for EM rescaling at test
time), and the synthetic document IDs (for filtering at test time).

In [52]:
X_train = np.concatenate(
    [head_datasets_syn[(l, h)][syn_train_idx] for l, h in top_k_heads], axis=1
)
y_train = syn_labels[syn_train_idx]

final_probe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0, class_weight=None, solver='lbfgs',
        max_iter=1000, random_state=42,
    ))
]).fit(X_train, y_train)

# Save probe + metadata for use in synthetic_probe_test.ipynb
probe_dir  = paths.trained_probe_dir(DATASET, JUDGE_MODEL)
probe_dir.mkdir(parents=True, exist_ok=True)
probe_path = probe_dir / 'head_probe.pkl'

probe_data = {
    'probe':            final_probe,
    'top_k_heads':      top_k_heads,
    'train_prevalence': float(y_train.mean()),
    'syn_document_ids': sorted(syn_df['document_id'].unique().tolist()),
    'judge_model':      JUDGE_MODEL,
    'dataset':          DATASET,
    'n_layers':         n_layers,
    'n_heads':          n_heads,
    'head_dim':         head_dim,
}
joblib.dump(probe_data, probe_path)

print(f'Probe saved  → {probe_path}')
print(f'  Top-{TOP_K} heads        : {top_k_heads}')
print(f'  Training prevalence : {y_train.mean():.3f}')
print(f'  Synthetic papers    : {len(probe_data["syn_document_ids"])}')

Probe saved  → /projectnb/mcnet/kevin/coastal/scholarlm/data/experiments/nfix/synthetic_probe/mistral-7b/trained_probe/head_probe.pkl
  Top-1 heads        : [(np.int64(14), np.int64(29))]
  Training prevalence : 0.333
  Synthetic papers    : 43


---
## Part II — Layer Output Probe (legacy)

Repeat the same CV → train protocol using the full residual-stream vector at
each layer (shape: `n_layers × hidden_size`) instead of individual attention
heads.  The best layer is selected by CV F1 and a final probe is trained on
the full synthetic training set.

### Build Layer Feature Matrices

In [ ]:
syn_layer_outputs = load_synthetic_layer_outputs(DATASET, JUDGE_MODEL, JUDGE_DATE_SYN)

_arr0_lo = np.array(syn_layer_outputs[str(syn_measurement_ids[0])], dtype=np.float32)
n_layers_lo, hidden_size = _arr0_lo.shape
print(f'Layer output shape : n_layers={n_layers_lo}, hidden_size={hidden_size}')

print('\nBuilding per-layer feature matrices (synthetic)...')
_all_syn_lo = {str(mid): np.array(syn_layer_outputs[str(mid)], dtype=np.float32)
               for mid in syn_measurement_ids}
layer_datasets_syn: dict[int, np.ndarray] = {
    l: np.stack([_all_syn_lo[str(mid)][l] for mid in syn_measurement_ids], axis=0)
    for l in range(n_layers_lo)
}
del _all_syn_lo
print(f'  Done. Shape: {layer_datasets_syn[0].shape}')

### Layer Selection via Cross-Validation

In [ ]:
probe_template_lo = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=0.1, class_weight='balanced', solver='lbfgs',
        max_iter=1000, random_state=42,
    ))
])

layer_scores_f1    = np.zeros(n_layers_lo)
layer_scores_auroc = np.zeros(n_layers_lo)

print(f'Scoring {n_layers_lo} layers on synthetic CV pool (n={len(syn_cv_idx)})...')
for l in range(n_layers_lo):
    X_cv = layer_datasets_syn[l][syn_cv_idx]
    (_, _, mean_auroc, _, mean_f1, _, _, _, _) = cv_score(
        probe_template_lo, X_cv, syn_labels_cv, kfold_cv
    )
    layer_scores_f1[l]    = mean_f1
    layer_scores_auroc[l] = mean_auroc
    if (l + 1) % 8 == 0:
        print(f'  Layer {l + 1}/{n_layers_lo} done')
print('Done.')

In [ ]:
# F1 by layer line plot — layer output probe
best_layer_lo = int(layer_scores_f1.argmax())
print(f'Best layer : {best_layer_lo}  '
      f'AUROC={layer_scores_auroc[best_layer_lo]:.4f}  '
      f'F1={layer_scores_f1[best_layer_lo]:.4f}')

fig, ax = plt.subplots(figsize=(3.5, 2.5))
ax.plot(range(n_layers_lo), layer_scores_f1, 'o-', color='#3984e0', ms=3, label='CV F1')
ax.plot(range(n_layers_lo), layer_scores_auroc, 's--', color='#888888', ms=2.5, lw=0.9, label='CV AUROC')
ax.axvline(best_layer_lo, color='#e07b39', lw=1.0, ls=':',
           label=f'Best: L{best_layer_lo} (F1={layer_scores_f1[best_layer_lo]:.3f})')
ax.set_xlabel('Layer')
ax.set_ylabel('Score')
ax.set_title(f'Layer output probe — CV score by layer ({DATASET}, {JUDGE_MODEL})')
ax.legend(fontsize=7)
ax.set_xlim(-0.5, n_layers_lo - 0.5)
fig.tight_layout()
#fig.savefig(FIGURES_DIR + f'synprobe_layer_f1_by_layer_{DATASET}_j-{JUDGE_MODEL}.pdf', bbox_inches='tight')

### Train Final Layer Probe

In [ ]:
X_train_lo = layer_datasets_syn[best_layer_lo][syn_train_idx]
y_train_lo = syn_labels[syn_train_idx]

final_probe_lo = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, class_weight=None, solver='lbfgs',
                               max_iter=1000, random_state=42))
]).fit(X_train_lo, y_train_lo)

train_acc = float((final_probe_lo.predict(X_train_lo) == y_train_lo).mean())
print(f'Layer {best_layer_lo} probe trained on {len(X_train_lo)} samples.')
print(f'  Training accuracy : {train_acc:.4f}')